# Robust Hierarchical Allocation — Synthetic Pre-registration Demo

**Synthetic data only.** This notebook executes a reduced factorial grid consistent with the pre-registered design in `RESEARCH_NOTE_UPGRADED.md`. No real-market claims.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

import pandas as pd
import numpy as np
from data.generate import make_synthetic_panel
from backtest.walk_forward import walk_forward
from metrics.evaluation import portfolio_stats, frobenius, effective_bets, block_bootstrap_sharpe

pd.set_option('display.float_format', lambda x: f'{x:.4f}')
print('Imports OK')

Imports OK


In [2]:
def run_cell(n_assets, corr_type, lookback, seed, cost_bps=5.0, n_periods=150):
    rets, meta = make_synthetic_panel(n_assets, n_periods, corr_type, lookback, seed)
    res = walk_forward(rets, meta['true_sigmas'], lookback, cost_bps=cost_bps)
    rows = []
    for m in res['returns'].columns:
        stats = portfolio_stats(res['returns'][m])
        lo, hi = block_bootstrap_sharpe(res['returns'][m], n_boot=80, seed=seed)
        rows.append({
            'n_assets': n_assets,
            'corr_type': corr_type,
            'lookback': lookback,
            'seed': seed,
            'method': m,
            'mean_frobenius': frobenius(res['errors'][m]),
            'eff_bets': effective_bets(res['weights'][m]),
            'sharpe_lo': lo,
            'sharpe_hi': hi,
            **stats,
        })
    return pd.DataFrame(rows)

# Reduced primary grid (still covers N/T, correlation structure, seeds)
cells = []
for n in [8, 20]:
    for corr in ['low', 'moderate', 'high']:
        for T in [36, 60]:
            for seed in [1, 2]:
                cells.append(run_cell(n, corr, T, seed))
df = pd.concat(cells, ignore_index=True)
print(f'Completed {len(df)} result rows')
df.head()

Completed 168 result rows


,n_assets,corr_type,lookback,seed,method,mean_frobenius,eff_bets,sharpe_lo,sharpe_hi,ann_return,ann_vol,sharpe,max_dd
0,8,low,36,1,equal_weight,NaN,8.0000,-0.8504,0.1711,-0.0391,0.0924,-0.4236,-0.4196
1,8,low,36,1,inverse_vol,NaN,7.2453,-0.7517,0.3413,-0.0237,0.0812,-0.2914,-0.3394
2,8,low,36,1,sample_mv,0.0047,4.3218,-0.4269,0.6336,0.0037,0.0771,0.0475,-0.2496
3,8,low,36,1,lw_mv,0.0032,6.0602,-0.6196,0.4911,-0.0095,0.0785,-0.1204,-0.2852
4,8,low,36,1,const_corr_mv,0.0030,4.7095,-0.3698,0.6256,0.0054,0.0737,0.0730,-0.2150


In [3]:
# Primary diagnostic: estimation error by N and method
err = df.groupby(['n_assets', 'method'])['mean_frobenius'].mean().unstack()
print('Mean Frobenius estimation error')
display(err.round(4))

Mean Frobenius estimation error


method,const_corr_mv,equal_weight,factor_mv,hrp,inverse_vol,lw_mv,sample_mv
n_assets,,,,,,,
8,0.0026,NaN,0.4765,0.0028,NaN,0.0028,0.0035
20,0.0050,NaN,0.7682,0.0072,NaN,0.0072,0.0078


In [4]:
# Stability diagnostics
stab = df.groupby('method')[['eff_bets', 'ann_vol', 'sharpe']].mean()
print('Average effective bets, volatility, Sharpe (net of 5 bps)')
display(stab.round(3))

Average effective bets, volatility, Sharpe (net of 5 bps)


,eff_bets,ann_vol,sharpe
method,,,
const_corr_mv,4.9630,0.0730,0.0150
equal_weight,14.0000,0.0990,-0.2620
factor_mv,3.4370,0.0820,-0.1580
hrp,10.8560,0.0840,-0.1660
inverse_vol,12.7510,0.0900,-0.1980
lw_mv,6.8800,0.0760,-0.1010
sample_mv,4.6060,0.0750,-0.0520


In [5]:
# Example of bootstrap intervals for one cell
one = df[(df.n_assets == 20) & (df.corr_type == 'high') & (df.lookback == 36)]
print('Sharpe 90% block-bootstrap intervals (N=20, high corr, T=36)')
display(one.groupby('method')[['sharpe', 'sharpe_lo', 'sharpe_hi']].mean().round(3))

Sharpe 90% block-bootstrap intervals (N=20, high corr, T=36)


,sharpe,sharpe_lo,sharpe_hi
method,,,
const_corr_mv,0.2060,-0.3300,0.7310
equal_weight,-0.0230,-0.5670,0.4920
factor_mv,0.1620,-0.4100,0.7690
hrp,0.0650,-0.5050,0.6050
inverse_vol,0.0190,-0.5360,0.5440
lw_mv,0.1400,-0.4040,0.6960
sample_mv,0.1350,-0.4210,0.7230


## Interpretation note (pre-registered)

Any ranking observed above is conditional on the synthetic DGP, the chosen N/T ratios, and the 5 bp cost. A method is not declared superior unless the pre-registered criteria in the research note are met on the full grid. Negative or mixed outcomes are expected and informative.